# RG → PDG comparison

Run the W33 mass‑synthesis RG pipeline, compare low‑energy mass predictions (m_t, m_b, m_τ) to PDG reference values, and visualize results. This notebook is lightweight and intended for reproducible checks and quick exploration.

Sections:
1. Environment & dependencies
2. Quick symbolic model sketch
3. Numerical RG run (solve_ivp)
4. PDG comparison & plots
5. Ingest Claude artifacts and quick diffs

Note: this notebook uses the repository's `scripts/` utilities (no external data fetch).

In [ ]:
# Section 1 — Environment & dependencies
import sys
import platform
import importlib
from pprint import pprint

import numpy as np
import matplotlib.pyplot as plt
from math import log

# optional heavy deps (import if available)
scipy = importlib.import_module('scipy') if importlib.util.find_spec('scipy') else None
sympy = importlib.import_module('sympy') if importlib.util.find_spec('sympy') else None

print(f"Python: {sys.version.split()[0]}  |  Platform: {platform.platform()}")
print(f"numpy: {np.__version__}")
print(f"scipy: {scipy.__name__ if scipy else 'not available'}")
print(f"sympy: {sympy.__name__ if sympy else 'not available'}")

np.random.seed(42)

## Section 2 — Symbolic model (sketch)
Using SymPy we show how the Lagrangian pieces could be declared and exported as numeric functions. This cell is illustrative only.

In [ ]:
if sympy:
    import sympy as sp
    x, t = sp.symbols('x t')
    phi = sp.Function('phi')(x, t)
    L_phi = 0.5 * sp.diff(phi, t)**2 - 0.5 * sp.diff(phi, x)**2 - sp.Rational(1,4) * phi**4
    print('Example symbolic Lagrangian (phi^4) declared')
    # derive Euler-Lagrange (symbolic)
    EL = sp.diff(sp.diff(L_phi, sp.diff(phi, t)), t) + sp.diff(sp.diff(L_phi, sp.diff(phi, x)), x) - sp.diff(L_phi, phi)
    print('Euler-Lagrange (symbolic) example:')
    sp.pretty_print(sp.simplify(EL))
else:
    print('sympy not available — skipping symbolic sketch')

## Section 3 — Numerical RG run (use repository utilities)
Run `derive_yukawas_from_triads()` → `integrate_rg()` and inspect the low‑energy masses.

In [ ]:
from scripts.w33_mass_synthesis import derive_yukawas_from_triads
from scripts.w33_rg_flow import integrate_rg

# derive GUT-scale Yukawas (model)
yuk = derive_yukawas_from_triads()
print('GUT-scale Yukawas (triad model):')
pprint(yuk)

# integrate down to MZ
out = integrate_rg(yuk)
print('\nRG prediction at M_Z:')
print('  y_t(MZ) =', out['y_t_MZ'])
print('  m_t(MZ) [GeV] ~', out['m_t_MZ_GeV'])
print('  m_b(MZ) [GeV] ~', out['m_b_MZ_GeV'])
print('  m_tau(MZ) [GeV] ~', out['m_tau_MZ_GeV'])

## Section 4 — PDG comparison & visualization
Compare model predictions to PDG reference values and plot a side-by-side bar chart.

In [ ]:
from scripts.w33_pdg_compare import compare_to_pdg

cmp = compare_to_pdg()
print('\nModel / PDG ratios:')
pprint(cmp['ratios_model_over_pdg'])

# bar plot: model vs PDG for top, bottom, tau
labels = ['top', 'bottom', 'tau']
model_vals = [cmp['prediction_MZ_GeV']['m_t_pred_GeV'], cmp['prediction_MZ_GeV']['m_b_pred_GeV'], cmp['prediction_MZ_GeV']['m_tau_pred_GeV']]
pdg_vals = [cmp['pdg_reference']['top_pole_GeV'], cmp['pdg_reference']['b_at_MZ_approx_GeV'], cmp['pdg_reference']['tau_mass_GeV']]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(7,4))
ax.bar(x - width/2, model_vals, width, label='model')
ax.bar(x + width/2, pdg_vals, width, label='PDG (ref)')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Mass (GeV)')
ax.set_title('W33 RG prediction vs PDG (representative)')
ax.legend()
plt.grid(axis='y', alpha=0.25)
plt.show()

## Section 5 — Ingest & compare Claude outputs
Load the most recent Claude artifact (if present) and compute a simple L2 difference on numeric fields.

In [ ]:
import json
from pathlib import Path

art_dir = Path('artifacts')
claude_files = sorted(art_dir.glob('*claude*json')) if art_dir.exists() else []
if claude_files:
    print('Found Claude artifacts — showing the latest:')
    data = json.loads(claude_files[-1].read_text())
    pprint(list(data.keys())[:12])
else:
    print('No Claude artifacts found in artifacts/. (workspace watcher not active in notebook)')

## Quick tests & CI checks
Run a tiny regression check: ensure `compare_to_pdg()` returns expected keys.

In [ ]:
# tiny regression check
assert 'prediction_MZ_GeV' in cmp
assert all(k in cmp['prediction_MZ_GeV'] for k in ['m_t_pred_GeV','m_b_pred_GeV','m_tau_pred_GeV'])
print('Sanity checks passed')